#  🛰️ PCA to Detect Dunes in Satellite Images


This challenge is based on a real-world data science problem: to detect dunes in satellite images. I worked on this problem to be able to detect dunes on Mars. My approach was to use computer vision (a subfield of deep-learning) with a model trained on Earth Dunes, and then deploy this algorithm to Martian satellite imagery. 

Here, I am giving you a much reduced version of my dataset, both in terms of number of image, and size of the images. The satellite images come from the **Sentinel 2** dataset, and were reduced to greyscale.  We will use conventional statistical machine learning to determine whether the image contains a dune (class '1') or not (class '0').

We will also use PCA to reduce the dimensionality of our dataset, and measure whether this helps computation and classification.

This exercise is a bit different then others, in the sense that you are not dealing purely with tabular data. So keep this in mind:
- each image is an observation (sample)
- each pixel's luminosity level is a feature

## Load dataset
The loader keeps filename quadrants of the same parent image together when splitting. It treats suffixes such as `_ulc` and `_urc` as related crops; confirm provenance before claiming independence between locations.

Reserve a grouped validation subset from the development images. Use validation for comparisons and test once after selecting a final model.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('../../Excercise_data/L07').resolve()))
from dunes_dataset import DunesDataLoader
from sklearn.model_selection import GroupShuffleSplit

In [2]:
loader = DunesDataLoader()
development_images, test_images, y_development, y_test = loader.get_data()
development_groups = loader.train_set['parent_image'].to_numpy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_index, val_index = next(splitter.split(development_images, groups=development_groups))
train_images = development_images[train_index]
val_images = development_images[val_index]
y_train = y_development.iloc[train_index].reset_index(drop=True)
y_val = y_development.iloc[val_index].reset_index(drop=True)
train_groups = development_groups[train_index]

# Exploring the dataset

First, look at the shape of the `train_images`:

In [3]:
#

Read the actual number of training images from the shape. Each has 64 x 64 pixels. Plot 25 images in a 5 x 5 grid with labels as titles.

<details><summary>Hint</summary>Use `plt.subplots` and `imshow(..., cmap='gray')`.</details>

In [4]:
#

# Preparing the feature matrices
For these estimators, represent each image as a row of 4096 pixel intensities.

1. Flatten `train_images`, `val_images` and `test_images` using `reshape`.
2. Divide by 255 to map the known pixel range to [0, 1]. This fixed conversion learns no sample statistics.

Name the results `X_train`, `X_val` and `X_test`.

In [5]:
# ADD YOUR CODE HERE

## Compress images with linear PCA
There are many features relative to the training observations. This can make estimation difficult but does not guarantee poor classification.

Fit `PCA(n_components=100, svd_solver='randomized', random_state=42)` on **X_train only**. Transform all three sets into `X_train_projected`, `X_val_projected` and `X_test_projected`. Keep the original feature matrices.

In [6]:
# ADD YOUR CODE HERE

## Thinking in terms of dimensionality reduction

It is important to think about what we have done above. We have reduced the images represented by the original 4096 pixels (or 4096 dimensions) into just 100 dimensions (the 100 first **principal components** of the images). Remember from your `Numerical Mathematics` and my own module that what we call components are directions of most variance of the dataset. 

Reducing the 4096 pixels  to describe each images into just 100 values is a gain in dimensionality by factor about 40!

**How does it work?**

- The pca has found to be the most representative directions of what distinguishes each images between each other with just 100 values for every image. 

- They are the directions of most variance. 

- You can access them in `pca.components_`

👉  Look at the first component of this array of components, and its shape

In [7]:
# ADD YOUR CODE HERE

The 100 component directions each have 4096 entries. Each image has 100 **scores**.

Use `pca.inverse_transform(X_train_projected)` to create `data_reconstructed`. This adds the training mean and reconstructs an approximation from the retained directions.

In [8]:
# ADD YOUR CODE HERE

👉 Plot the 1st picture of the reconstructed dataset, and compare it with the original one. 

<details>
    <summary>💡Hint</summary>

You'll have to reshape the flattened data into an "image" with the appropriate pixel dimensions (64x64)
</details>

In [9]:
# ADD YOUR CODE HERE

## Investigate your Principal Components
👉 Image-plot the "mean" satellite image in the dataset

<details>
    <summary>💡Hint</summary>


You can use `pca.mean_`
</details>


In [10]:
# ADD YOUR CODE HERE

👉 Access your first PC. What's its shape? Print it as pd.Series or NDarray. What does each values represents?

In [11]:
# ADD YOUR CODE HERE

Each direction reshapes into 64 x 64 pixel weights. The first maximises projected training variance; later directions explain remaining variance while staying orthogonal. A direction is a pattern across pixels, not a region selected as important for dune classification.

👉 Image-Plot the **5 first** principal components, as well as the **last** one.
Do you see more intuitively what PC are? 

In [12]:
# ADD YOUR CODE HERE

An image is **approximately** represented by the training mean plus a weighted sum of the retained 100 directions. Omitted components explain reconstruction error.

## How to Choose the Number of Components?

In practice, it is very important to find how many components are needed to describe the data without losing too much information. This can be determined visually by plotting the cumulative sum of `explained_variance_ratio_` as a function of the number of components.
 
👉 Plot it below for the first 100 components

In [13]:
# ADD YOUR CODE HERE

Read the cumulative curve rather than assuming a threshold is reached. What variance fraction do 100 components retain?

Find the minimum number retaining at least 95% of **training** variance and name it `minimal_pc_count`. If 100 components never reach 95%, fit `PCA(n_components=0.95, svd_solver='full')` on `X_train` and read `n_components_`. That full decomposition can take longer.

In [14]:
# ADD YOUR CODE HERE

## PCA as feature engineering
Use retained PCA scores as classifier inputs. Preserving 95% of pixel variance does not guarantee preserving 95% of class information.

### Transform using the training fit
Fit PCA with `n_components=minimal_pc_count` on `X_train` only. Transform all three sets into `X_train_red`, `X_val_red` and `X_test_red`.

In [15]:
# ADD YOUR CODE HERE

In [16]:
# ADD YOUR CODE HERE

# Testing the impact of PCA on dune classification

Here we will do two things:

* 1️⃣ We will check how `PCA` impacts our training time
* 2️⃣ We will check how `PCA` impacts our predictions

We will limit ourselves to a simple linear classification algorithm (`LogisticRegression`).

First, using the `%%timeit` magic function, train a `LogisticRegression` model with the following parameters:
* `max_iter = 5000`
* Trained on `X_train` and `y_train` (the original features)

Take note of the training time!

In [17]:
# ADD YOUR CODE HERE

In [18]:
# ADD YOUR CODE HERE

Now, also using the `%%timeit` magic function, train a `LogisticRegression` model (call it `pca_lr`) but this time with the following parameters:
* `max_iter = 5000`
* Trained on `X_train_red` and `y_train` (the PCA features)

Take note of the training time again.

In [19]:
# ADD YOUR CODE HERE

### Compare training time
Record the ratio of full-feature to PCA-feature classifier fitting times in the same units. Is either faster? Report PCA fitting time separately: classifier-only timing omits representation construction.

In [20]:
# ADD YOUR CODE HERE

### Compare validation accuracy
`%%timeit` repeats computation to measure runtime; it does **not** perform cross-validation.

Fit fresh `full_model` and `pca_model` logistic regressions outside timing cells. Compare accuracy on `X_val` and `X_val_red` using `y_val`. Keep test data out of this comparison.

In [21]:
# ADD YOUR CODE HERE

Explain the observed difference without assuming PCA must help. Which discarded directions could still contain class information? How might another grouped train/validation split affect the conclusion?

## Optional: model selection with PCA and a random forest
Build a Pipeline containing PCA and RandomForestClassifier. Compare a small component-count/depth grid with `GridSearchCV` on development images. Use `GroupKFold(n_splits=3)` and pass `groups=development_groups` to `fit`. Fit PCA inside the pipeline so it stays inside each fold.

After selecting one model using development data, evaluate it **once** on `X_test` and `y_test`. Report the grouping assumption and why this score does not establish transfer to Mars.

In [22]:
# ADD YOUR CODE HERE

In [23]:
# ADD YOUR CODE HERE